# WSI Processing Pipeline Tutorial

This notebook demonstrates end-to-end usage of the WSI Processing Pipeline for computational pathology.

## Overview

The WSI Processing Pipeline enables processing of Whole Slide Images (WSI) in clinical formats:
- Reading WSI files (.svs, .tiff, .ndpi, DICOM)
- Extracting patches at multiple magnification levels
- Detecting tissue regions and filtering background
- Generating feature embeddings using pretrained encoders
- Caching features in HDF5 format
- Training models with processed features

## Table of Contents

1. [Setup and Installation](#setup)
2. [Reading WSI Files](#reading)
3. [Extracting Patches](#patches)
4. [Tissue Detection](#tissue)
5. [Feature Generation](#features)
6. [Caching Features](#caching)
7. [Batch Processing](#batch)
8. [Loading Processed Features](#loading)
9. [Training with Processed Features](#training)

## 1. Setup and Installation <a name="setup"></a>

First, let's import the required libraries and check that everything is installed correctly.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

from src.data.wsi_pipeline import (
    WSIReader,
    PatchExtractor,
    TissueDetector,
    FeatureGenerator,
    FeatureCache,
    BatchProcessor,
    ProcessingConfig,
    QualityControl,
)

print("✓ All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Reading WSI Files <a name="reading"></a>

Let's start by reading a WSI file and inspecting its properties.

In [ ]:
# Path to your WSI file (update this path)
wsi_path = "path/to/your/slide.svs"

# Open the WSI file
with WSIReader(wsi_path) as reader:
    # Get slide properties
    dimensions = reader.dimensions
    level_count = reader.level_count
    level_dimensions = reader.level_dimensions
    level_downsamples = reader.level_downsamples
    
    # Get metadata
    magnification = reader.get_magnification()
    mpp = reader.get_mpp()
    scanner_info = reader.get_scanner_info()
    
    print("Slide Properties:")
    print(f"  Dimensions (level 0): {dimensions[0]} x {dimensions[1]} pixels")
    print(f"  Pyramid levels: {level_count}")
    print(f"  Magnification: {magnification}x" if magnification else "  Magnification: Not available")
    print(f"  MPP: {mpp}" if mpp else "  MPP: Not available")
    print(f"  Scanner: {scanner_info.get('model', 'Unknown')}")
    print()
    print("Pyramid Levels:")
    for i, (dims, downsample) in enumerate(zip(level_dimensions, level_downsamples)):
        print(f"  Level {i}: {dims[0]} x {dims[1]} (downsample: {downsample:.2f}x)")
    
    # Get a thumbnail for visualization
    thumbnail = reader.get_thumbnail(size=(512, 512))
    
# Display thumbnail
plt.figure(figsize=(8, 8))
plt.imshow(thumbnail)
plt.title("Slide Thumbnail")
plt.axis('off')
plt.show()

## 3. Extracting Patches <a name="patches"></a>

Now let's extract patches from the WSI at a specific magnification level.

In [ ]:
# Create patch extractor
extractor = PatchExtractor(
    patch_size=256,
    stride=256,  # Non-overlapping patches
    level=0,     # Highest resolution
)

with WSIReader(wsi_path) as reader:
    # Generate coordinates for a small region (for demo purposes)
    # In practice, you would use the full slide dimensions
    demo_dimensions = (5000, 5000)  # Small region for demo
    coordinates = extractor.generate_coordinates(demo_dimensions)
    
    print(f"Generated {len(coordinates)} patch coordinates")
    print(f"First 5 coordinates: {coordinates[:5]}")
    
    # Extract a few patches for visualization
    sample_patches = []
    sample_coords = coordinates[:9]  # Extract 9 patches for 3x3 grid
    
    for coord in sample_coords:
        patch = extractor.extract_patch(reader, coord)
        sample_patches.append(patch)

# Visualize extracted patches
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for i, (ax, patch, coord) in enumerate(zip(axes.flat, sample_patches, sample_coords)):
    ax.imshow(patch)
    ax.set_title(f"Patch at ({coord[0]}, {coord[1]})")
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Tissue Detection <a name="tissue"></a>

Let's detect tissue regions and filter out background patches.

In [ ]:
# Create tissue detector
detector = TissueDetector(
    tissue_method="otsu",
    tissue_threshold=0.5,  # Require 50% tissue coverage
)

with WSIReader(wsi_path) as reader:
    # Generate tissue mask at low resolution
    tissue_mask = detector.generate_tissue_mask(reader)
    
    print(f"Tissue mask shape: {tissue_mask.shape}")
    print(f"Tissue coverage: {tissue_mask.mean() * 100:.2f}%")
    
    # Check tissue content in sample patches
    tissue_patches = []
    background_patches = []
    
    for patch in sample_patches:
        is_tissue = detector.is_tissue_patch(patch)
        tissue_pct = detector.calculate_tissue_percentage(patch)
        
        if is_tissue:
            tissue_patches.append((patch, tissue_pct))
        else:
            background_patches.append((patch, tissue_pct))
    
    print(f"\nTissue patches: {len(tissue_patches)}")
    print(f"Background patches: {len(background_patches)}")

# Visualize tissue mask
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(thumbnail)
axes[0].set_title("Original Thumbnail")
axes[0].axis('off')

axes[1].imshow(tissue_mask, cmap='gray')
axes[1].set_title("Tissue Mask")
axes[1].axis('off')

plt.tight_layout()
plt.show()

# Show tissue vs background patches
if tissue_patches and background_patches:
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    
    patch, pct = tissue_patches[0]
    axes[0].imshow(patch)
    axes[0].set_title(f"Tissue Patch ({pct*100:.1f}% tissue)")
    axes[0].axis('off')
    
    patch, pct = background_patches[0]
    axes[1].imshow(patch)
    axes[1].set_title(f"Background Patch ({pct*100:.1f}% tissue)")
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()

## 5. Feature Generation <a name="features"></a>

Let's generate feature embeddings from tissue patches using a pretrained encoder.

In [ ]:
# Create feature generator
generator = FeatureGenerator(
    encoder_name="resnet50",
    pretrained=True,
    device="cuda" if torch.cuda.is_available() else "cpu",
    batch_size=8,
)

print(f"Feature generator initialized")
print(f"  Encoder: {generator.encoder_name}")
print(f"  Device: {generator.device}")
print(f"  Feature dimension: {generator.feature_dim}")

# Convert tissue patches to tensor
if tissue_patches:
    patches_array = np.stack([p[0] for p in tissue_patches])
    patches_tensor = torch.from_numpy(patches_array).permute(0, 3, 1, 2).float() / 255.0
    
    print(f"\nPatches tensor shape: {patches_tensor.shape}")
    
    # Extract features
    features = generator.extract_features(patches_tensor)
    
    print(f"Features shape: {features.shape}")
    print(f"Feature statistics:")
    print(f"  Mean: {features.mean():.4f}")
    print(f"  Std: {features.std():.4f}")
    print(f"  Min: {features.min():.4f}")
    print(f"  Max: {features.max():.4f}")
    
    # Visualize feature distribution
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.hist(features.flatten().cpu().numpy(), bins=50)
    plt.title("Feature Value Distribution")
    plt.xlabel("Feature Value")
    plt.ylabel("Count")
    
    plt.subplot(1, 2, 2)
    plt.imshow(features.cpu().numpy(), aspect='auto', cmap='viridis')
    plt.title("Feature Heatmap")
    plt.xlabel("Feature Dimension")
    plt.ylabel("Patch Index")
    plt.colorbar()
    
    plt.tight_layout()
    plt.show()

## 6. Caching Features <a name="caching"></a>

Let's save the extracted features to HDF5 format for reuse.

In [ ]:
# Create feature cache
cache = FeatureCache(
    cache_dir="features",
    compression="gzip",
)

# Save features
slide_id = "demo_slide"
coordinates_array = np.array(sample_coords[:len(tissue_patches)])

metadata = {
    "slide_id": slide_id,
    "patient_id": "demo_patient",
    "magnification": magnification,
    "mpp": mpp[0] if mpp else None,
    "patch_size": 256,
    "stride": 256,
    "level": 0,
    "encoder_name": "resnet50",
    "num_patches": len(tissue_patches),
}

feature_file = cache.save_features(
    slide_id=slide_id,
    features=features.cpu().numpy(),
    coordinates=coordinates_array,
    metadata=metadata,
)

print(f"Features saved to: {feature_file}")
print(f"File size: {feature_file.stat().st_size / 1024:.2f} KB")

# Verify the cached file
is_valid = cache.validate(slide_id)
print(f"File validation: {'✓ Valid' if is_valid else '✗ Invalid'}")

# Load features back
loaded_data = cache.load_features(slide_id)

print(f"\nLoaded data:")
print(f"  Features shape: {loaded_data['features'].shape}")
print(f"  Coordinates shape: {loaded_data['coordinates'].shape}")
print(f"  Metadata keys: {list(loaded_data['metadata'].keys())}")

# Verify features match
features_match = np.allclose(features.cpu().numpy(), loaded_data['features'])
print(f"  Features match: {'✓ Yes' if features_match else '✗ No'}")

## 7. Batch Processing <a name="batch"></a>

Let's process a complete slide using the BatchProcessor.

In [ ]:
# Create processing configuration
config = ProcessingConfig(
    patch_size=256,
    stride=256,
    level=0,
    tissue_method="otsu",
    tissue_threshold=0.5,
    encoder_name="resnet50",
    encoder_pretrained=True,
    batch_size=32,
    cache_dir="features",
    compression="gzip",
    num_workers=1,
    max_retries=3,
    blur_threshold=100.0,
    min_tissue_coverage=0.1,
)

# Initialize batch processor
processor = BatchProcessor(
    config=config,
    num_workers=1,
    gpu_ids=None,  # Auto-detect
)

print("Batch processor initialized")
print(f"  Configuration: {config}")

# Process single slide
print(f"\nProcessing slide: {wsi_path}")
print("This may take several minutes...")

result = processor.process_slide(
    wsi_path=wsi_path,
    output_dir="data/processed",
)

if result.success:
    print(f"\n✓ Processing completed successfully!")
    print(f"  Slide ID: {result.slide_id}")
    print(f"  Number of patches: {result.num_patches}")
    print(f"  Processing time: {result.processing_time:.2f}s")
    print(f"  Feature file: {result.feature_file}")
    
    if result.qc_metrics:
        print(f"  Quality metrics:")
        for key, value in result.qc_metrics.items():
            print(f"    • {key}: {value}")
else:
    print(f"\n✗ Processing failed: {result.error_message}")

## 8. Loading Processed Features <a name="loading"></a>

Let's load the processed features using CAMELYONSlideDataset.

In [ ]:
from src.datasets.camelyon_slide_dataset import CAMELYONSlideDataset

# Create dataset from processed features
# Note: You need to have a slide_index.json file
# This can be generated using processor.generate_slide_index()

try:
    dataset = CAMELYONSlideDataset(
        data_dir="data/processed",
        split="train",
        slide_index_path="data/processed/slide_index.json",
    )
    
    print(f"Dataset loaded successfully")
    print(f"  Number of slides: {len(dataset)}")
    
    # Load a sample
    if len(dataset) > 0:
        sample = dataset[0]
        
        print(f"\nSample data:")
        print(f"  Features shape: {sample['features'].shape}")
        print(f"  Coordinates shape: {sample['coordinates'].shape}")
        print(f"  Label: {sample['label']}")
        print(f"  Slide ID: {sample['slide_id']}")
        
        # Visualize feature distribution
        plt.figure(figsize=(10, 4))
        plt.subplot(1, 2, 1)
        plt.hist(sample['features'].flatten(), bins=50)
        plt.title("Feature Distribution")
        plt.xlabel("Feature Value")
        plt.ylabel("Count")
        
        plt.subplot(1, 2, 2)
        plt.scatter(sample['coordinates'][:, 0], sample['coordinates'][:, 1], s=1, alpha=0.5)
        plt.title("Patch Coordinates")
        plt.xlabel("X")
        plt.ylabel("Y")
        plt.gca().invert_yaxis()
        
        plt.tight_layout()
        plt.show()
        
except Exception as e:
    print(f"Error loading dataset: {e}")
    print("Make sure you have generated a slide index using processor.generate_slide_index()")

## 9. Training with Processed Features <a name="training"></a>

Finally, let's demonstrate how to use the processed features for training a MIL model.

In [ ]:
from torch.utils.data import DataLoader
from src.models.attention_mil import AttentionMIL

# Create data loader
try:
    dataloader = DataLoader(
        dataset,
        batch_size=1,  # MIL models typically use batch_size=1
        shuffle=True,
        num_workers=0,
    )
    
    # Create MIL model
    model = AttentionMIL(
        input_dim=2048,  # ResNet-50 feature dimension
        hidden_dim=256,
        num_classes=2,
    )
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    
    print(f"Model created: {model.__class__.__name__}")
    print(f"  Input dimension: 2048")
    print(f"  Hidden dimension: 256")
    print(f"  Number of classes: 2")
    print(f"  Device: {device}")
    
    # Run a forward pass
    model.eval()
    with torch.no_grad():
        batch = next(iter(dataloader))
        features = batch['features'].to(device)
        
        print(f"\nForward pass:")
        print(f"  Input shape: {features.shape}")
        
        output = model(features)
        
        print(f"  Output shape: {output.shape}")
        print(f"  Predictions: {output.softmax(dim=1)}")
    
    print(f"\n✓ Model inference successful!")
    print(f"\nYou can now train the model using:")
    print(f"  python src/training/train_camelyon.py --data-dir data/processed")
    
except Exception as e:
    print(f"Error: {e}")
    print("Make sure you have a valid dataset with processed features")

## Summary

In this tutorial, we covered:

1. **Reading WSI files** - Opening slides in multiple formats and extracting metadata
2. **Extracting patches** - Sampling patches at different magnification levels
3. **Tissue detection** - Filtering background regions using Otsu thresholding
4. **Feature generation** - Extracting embeddings using pretrained encoders
5. **Caching features** - Storing features in HDF5 format for reuse
6. **Batch processing** - Processing complete slides with the BatchProcessor
7. **Loading features** - Using CAMELYONSlideDataset to load processed features
8. **Training models** - Using processed features with MIL models

## Next Steps

- Process your own WSI slides using the batch processing script
- Experiment with different encoders and configurations
- Train MIL models on your processed features
- Evaluate model performance on test slides

For more information, see:
- `src/data/wsi_pipeline/README.md` - Complete documentation
- `examples/process_single_slide.py` - Single slide processing script
- `examples/process_batch_slides.py` - Batch processing script
- `examples/wsi_pipeline_config.yaml` - Configuration file example